In [0]:
%sql
create catalog if not exists investment_pyspark;
use catalog investment_pyspark;

create schema if not exists bronze;
create volume if not exists bronze.landing;

In [0]:
from pyspark.sql.functions import current_timestamp, col
import re

In [0]:
display(
    dbutils.fs.ls("/Volumes/investment_pyspark/bronze/landing/holding/holdings.csv/")
)

In [0]:
# 1. Source Path (Where your uploaded holdings.csv lives)
source_path = "/Volumes/investment_pyspark/bronze/landing/holding/"

# 2. Schema Path (Databricks will create this '_schemas' folder automatically)
schema_path = "/Volumes/investment_pyspark/bronze/landing/holding/holding_schemas/"

# 3. Checkpoint Path (Databricks will create this '_checkpoints' folder automatically)
checkpoint_path = (
    "/Volumes/investment_pyspark/bronze/landing/holding/holding_checkpoints/holdings"
)
#dbutils.fs.rm(schema_path, True)
#dbutils.fs.rm(checkpoint_path, True)

def clean_column_name(column_name):
    column_name = column_name.strip()
    column_name = re.sub(r'[^a-zA-Z0-9_]', '_', column_name)
    column_name = re.sub(r'_+', '_', column_name)
    return column_name.strip('_')


In [0]:
# Fix: pathGlobFilter ensures Auto Loader only processes CSV files,
# ignoring checkpoint/schema files inside the source directory
# Clean up directories from previous fix (outside holding/)
#dbutils.fs.rm("/Volumes/investment_pyspark/bronze/landing/holding_schemas", True)
#dbutils.fs.rm("/Volumes/investment_pyspark/bronze/landing/holding_checkpoints", True)

# Drop existing table for a clean restart
#spark.sql("DROP TABLE IF EXISTS investment_pyspark.bronze.holdings_raw")

df_bronze = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("pathGlobFilter", "*.csv")
    .option("header", "true")
    #.option("inferSchema", "true")
    .load(source_path)
)
df_cleaned = df_bronze.toDF(*[clean_column_name(c) for c in df_bronze.columns])
df_transformed = df_cleaned.withColumn(
    "_ingestion_timestamp", current_timestamp()
).withColumn("_source_file",col("_metadata.file_path"))
query = (df_transformed.writeStream.format("delta")
         .option("checkpointLocation", checkpoint_path)
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable("investment_pyspark.bronze.holdings_raw"))
query.awaitTermination()
print(f"Stream completed. Processed {query.lastProgress}")


In [0]:
display(df_cleaned.columns)

## Normalizeing only column names

In [0]:
#df_bronze.write.format("delta").saveAsTable("investment_pyspark.bronze.holdings_raw")

In [0]:
df_check = spark.read.table("investment_pyspark.bronze.holdings_raw")
display(df_check)

In [0]:
df_check.printSchema()

In [0]:
%sql
select * from investment_pyspark.bronze.holdings_raw;